In [67]:
# %pip install import-ipynb
# import import_ipynb

In [68]:
# General Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import os

# Modelling Libraries
import statsmodels.api as sm

from sklearn.preprocessing import StandardScaler, MinMaxScaler

from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.inspection import permutation_importance
from sklearn.feature_selection import RFE
from sklearn.metrics import mean_squared_error, r2_score
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [92]:
merged_df = pd.read_csv("merged_df.csv", encoding="utf-8-sig")
qgis_df = pd.read_csv("QGIS_df.csv", encoding="utf-8-sig")
general_df = pd.read_csv("general_df.csv", encoding="utf-8-sig")
cmci_df = pd.read_csv(os.path.join("Independent Data", "CMCI_2023_Shortlist.csv"))
atm_df = pd.read_csv("20250305 Updated Properties with ATM.csv")

In [112]:
# Price not included
detail_cols =['File', 'URL', 'SKU', 'Title', 'Category', "Property Type", 'Description', 'Agent_Name', 'Agent_Link', 'Agent_Verification']
loc_cols = ['Latitude', 'Longitude', 'Location', "Matched_City"]
value_cols = ['Num_Bedrooms', 'Num_Bathrooms', 'Floor_Area', 'Land_Area']
count_cols = ["LRTHubDist", 'BusCount', 'HospitalCount', 'MallCount', 'SchoolCount', "ATMCount"]
distance_cols = ["LRTHubDist", 'BusHubDist', 'HospitalHubDist', 'MallHubDist', 'SchoolHubDist', "ATMHubDist"]
# cmci_cols = ['employment_generation', 'financial_deepening', 'local_economy_growth', 'local_economy_size', 'presence_of_business_and_professional_organizations', 'safety_compliant_business']

#vif adjusted
# cmci_cols = ['local_economy_growth', 'local_economy_size', 'presence_of_business_and_professional_organizations']

#remove if many 0
cmci_cols = ['employment_generation', 'financial_deepening', 'local_economy_size', 'safety_compliant_business']

In [ ]:
cmci_df[(cmci_df[cmci_cols] == 0).any(axis=1)]

,PROVINCE / LGU,employment_generation,financial_deepening,local_economy_growth,local_economy_size,presence_of_business_and_professional_organizations,safety_compliant_business


In [ ]:
def X_subset(df, count_distance, w_cmci):        
    if count_distance == "count":
        cols = value_cols + count_cols
    if count_distance == "distance":
        cols = value_cols + distance_cols
    if w_cmci == True:
        cols = cols + cmci_cols
    return df[cols]

In [116]:
def calculate_vif(df):
    vif_data = pd.DataFrame()
    vif_data["Feature"] = df.columns
    vif_data["VIF"] = [variance_inflation_factor(df.values, i) for i in range(df.shape[1])]
    return vif_data

calculate_vif(cmci_df[cmci_cols])

,Feature,VIF
0,employment_generation,5.113695
1,financial_deepening,3.742688
2,local_economy_size,2.737750
3,safety_compliant_business,4.951797


# Preprocessing

In [ ]:
def normalize_data(df):
    # scaler = MinMaxScaler()
    scaler = StandardScaler()
    df_c = df.copy()
    numeric_cols = df_c.select_dtypes(include=['number']).columns
    df_c[numeric_cols] = scaler.fit_transform(df_c[numeric_cols])

    return df_c

def remove_outliers_iqr(df, feature):
    Q1 = df[feature].quantile(0.25)
    Q3 = df[feature].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    df_filtered = df[(df[feature] >= lower_bound) & (df[feature] <= upper_bound)]
    
    return df_filtered

def remove_outliers_iqr_list(df, feature_list):
    df_c = df.copy()
    for feature in feature_list:
        df_c = remove_outliers_iqr(df_c, feature)
        
    return df_c

def process_data(df, reduce, normalize):
    if reduce == True:
        feature_list = ["LRTHubDist", "Num_Bedrooms", "Num_Bathrooms", "Floor_Area", "Land_Area", "Price"]
        df = remove_outliers_iqr_list(df, feature_list)
    if normalize == True:
        df = normalize_data(df)
    return df

In [119]:
print(len(merged_df))
print(len(process_data(merged_df, True, True)))

72019
51863


# Models

In [ ]:
def basic_model(df, count_distance, remove_outliers, normalize_data):
    df = process_data(df, remove_outliers, normalize_data)
    
    y = df['Price']
    X = X_subset(df, count_distance, True)
    X = sm.add_constant(X)

    model = sm.OLS(y, X).fit()

    return model.summary()

In [ ]:
def basic_model2(df, count_distance, remove_outliers, normalize_data):
    df = process_data(df, remove_outliers, normalize_data)
    
    y = df['Price']
    X = X_subset(df, count_distance, True)

    model = LinearRegression()
    model.fit(X, y)

    mse = -cross_val_score(model, X, y, scoring='neg_mean_squared_error', cv=5).mean()
    r2 = cross_val_score(model, X, y, scoring='r2', cv=5).mean()

    coef_df = pd.DataFrame({"Feature": X.columns, "Coefficient": model.coef_})
    coef_df["Coefficient"] = coef_df["Coefficient"].apply(lambda x: f"{x:.4f}")

    print("\n===== Linear Regression Results =====")
    print(f"R² Score (cross-validated): {r2:.4f}")
    print(f"Mean Squared Error (MSE): {mse:.4f}")
    print("\nCoefficients:\n", coef_df.to_string(index=False))

In [ ]:
def cv_new(df, count_distance, remove_outliers, normalize_data, n_splits=5):
    df = process_data(df, remove_outliers, normalize_data)

    y = df['Price']
    X = X_subset(df, count_distance, True)
    X = sm.add_constant(X)  # Add constant for intercept

    model = LinearRegression()
    OLS_score = -cross_val_score(model, X, y, scoring = 'neg_mean_squared_error', cv=5).mean() # Average score
    print(f" OLS: \n MSE: {OLS_score}")


In [ ]:
def cv_model(df, count_distance, remove_outliers, normalize_data):
    df = process_data(df, remove_outliers, normalize_data)
    y = df['Price']
    X = X_subset(df, count_distance, True)
    X = sm.add_constant(X)

    # # Show summary per fold
    # kf = KFold(n_splits=5)
    # mse_scores = []
    # r2_scores = []

    # fold = 1
    # for train_index, test_index in kf.split(X):
    #     X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    #     y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    #     model = sm.OLS(y_train, X_train).fit()

    #     y_pred = model.predict(X_test)
    #     mse = np.mean((y_pred - y_test) ** 2)
    #     r2 = model.rsquared

    #     mse_scores.append(mse)
    #     r2_scores.append(r2)

    #     print(f"Fold {fold} Summary:")
    #     print(model.summary())
    #     fold += 1

    # mean_mse = np.mean(mse_scores)
    # mean_r2 = np.mean(r2_scores)

    # # Average metrics across folds
    # print(f"Average MSE across folds: {mean_mse:.2f}")
    # print(f"Average R² across folds: {mean_r2:.2f}")

    # return mean_mse, mean_r2

    model = LinearRegression()

    scores = cross_val_score(model, X, y, cv=5, scoring='neg_mean_squared_error')
    mean_mse = -np.mean(scores)  # Convert to positive MSE    
    r2_scores = cross_val_score(model, X, y, cv=5, scoring='r2')
    mean_r2 = np.mean(r2_scores)

    print(f"Average Mean Squared Error (MSE) across folds: {mean_mse:.2f}")
    print(f"Average R-squared (R²) across folds: {mean_r2:.2f}")

    return mean_mse, mean_r2

In [ ]:
def cv_model2(df, count_distance, remove_outliers, normalize_data):
    df = process_data(df, remove_outliers, normalize_data)
    y = df['Price']
    X = X_subset(df, count_distance, True)
    X = sm.add_constant(X)  # Add constant for intercept
    kf = KFold(n_splits=5, shuffle=True, random_state=42)  # Shuffle for randomness

    mse_scores = []
    r2_scores = []
    coef_matrix = np.zeros((kf.get_n_splits(), X.shape[1]))  # Store coefficients
    p_value_matrix = np.zeros((kf.get_n_splits(), X.shape[1]))  # Store p-values

    for fold, (train_index, test_index) in enumerate(kf.split(X)):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]

        model = sm.OLS(y_train, X_train).fit()
        y_pred = model.predict(X_test)

        # Store performance metrics
        mse_scores.append(mean_squared_error(y_test, y_pred))
        r2_scores.append(r2_score(y_test, y_pred))  # FIXED: Now uses test-set R²

        # Store coefficients and p-values
        coef_matrix[fold, :] = model.params.values
        p_value_matrix[fold, :] = model.pvalues.values

    # Compute average values
    avg_mse = np.mean(mse_scores)
    avg_r2 = np.mean(r2_scores)
    avg_coefs = np.mean(coef_matrix, axis=0)
    avg_p_values = np.mean(p_value_matrix, axis=0)

    # Create results DataFrame
    results_df = pd.DataFrame({
        "Feature": X.columns,
        "Average Coefficient": avg_coefs,
        "Average P>|t|": avg_p_values
    }).sort_values(by="Average P>|t|", ascending=True)  # Sort by significance

    results_df["Average Coefficient"] = results_df["Average Coefficient"].round(4)
    results_df["Average P>|t|"] = results_df["Average P>|t|"].round(4)

    # Print results
    print("\n===== Cross-Validation OLS Results =====")
    print(f"Average Mean Squared Error (MSE): {avg_mse:.4f}")
    print(f"Average R²: {avg_r2:.4f}")  # FIXED Negative R² issue
    print("\nCoefficients:\n", results_df.to_string(index=False))

    return results_df

# Example Usage
# results = cv_model2(general_df, "count", True, True)

In [ ]:
def cv_model2_adj(df, count_distance, remove_outliers, normalize_data):
    df = process_data(df, remove_outliers, normalize_data)
    y = df['Price']
    X = X_subset(df, count_distance, True)
    X = sm.add_constant(X)  # Add constant for intercept
    kf = KFold(n_splits=5, shuffle=True, random_state=42)  # Shuffle for randomness

    mse_scores = []
    r2_scores = []
    coef_matrix = np.zeros((kf.get_n_splits(), X.shape[1]))  # Store coefficients
    p_value_matrix = np.zeros((kf.get_n_splits(), X.shape[1]))  # Store p-values

    for fold, (train_index, test_index) in enumerate(kf.split(X)):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]

        model = sm.OLS(y_train, X_train).fit()
        y_pred = model.predict(X_test)

        # Store performance metrics
        mse_scores.append(mean_squared_error(y_test, y_pred))
        r2_scores.append(r2_score(y_test, y_pred))  # FIXED: Now uses test-set R²

        # Store coefficients and p-values
        coef_matrix[fold, :] = model.params.values
        p_value_matrix[fold, :] = model.pvalues.values

    # Compute average values
    avg_mse = np.mean(mse_scores)
    avg_r2 = np.mean(r2_scores)
    avg_coefs = np.mean(coef_matrix, axis=0)
    avg_p_values = np.mean(p_value_matrix, axis=0)

    # Create results DataFrame
    results_df = pd.DataFrame({
        "Feature": X.columns,
        "Average Coefficient": avg_coefs,
        "Average P>|t|": avg_p_values
    }).sort_values(by="Average P>|t|", ascending=True)  # Sort by significance

    results_df["Average Coefficient"] = results_df["Average Coefficient"].round(4)
    results_df["Average P>|t|"] = results_df["Average P>|t|"].round(4)

    # Print results
    print("\n===== Cross-Validation OLS Results =====")
    print(f"Average Mean Squared Error (MSE): {avg_mse:.4f}")
    print(f"Average R²: {avg_r2:.4f}")  # FIXED Negative R² issue
    print("\nCoefficients:\n", results_df.to_string(index=False))

    return results_df

# Example Usage
# results = cv_model2(general_df, "count", True, True)

In [106]:
from sklearn.model_selection import cross_validate

def cv_model3(df, count_distance, remove_outliers, normalize_data):
    df = process_data(df, remove_outliers, normalize_data)
    y = df['Price']
    X = X_subset(df, count_distance, True)
    X = sm.add_constant(X)  # Add constant for intercept

    model = LinearRegression()

    # Perform cross-validation
    cv_results = cross_validate(
        model, X, y, cv=5, 
        scoring=['neg_mean_squared_error', 'r2'], 
        return_estimator=True
    )

    avg_mse = -np.mean(cv_results['test_neg_mean_squared_error'])  # Convert to positive MSE
    avg_r2 = np.mean(cv_results['test_r2'])

    # Extract coefficients from each fold and compute average
    coef_matrix = np.array([estimator.coef_ for estimator in cv_results['estimator']])
    avg_coefs = np.mean(coef_matrix, axis=0)

    # Train final model on full dataset to get p-values
    final_model = sm.OLS(y, X).fit()
    final_p_values = final_model.pvalues.values

    # Create results DataFrame
    results_df = pd.DataFrame({
        "Feature": X.columns,
        "Average Coefficient": avg_coefs,
        "Final P>|t|": final_p_values
    }).sort_values(by="Final P>|t|", ascending=True)  # Sort by significance

    results_df["Average Coefficient"] = results_df["Average Coefficient"].round(4)
    results_df["Final P>|t|"] = results_df["Final P>|t|"].round(4)

    # Print results
    print("\n===== Cross-Validation Linear Regression Results =====")
    print(f"Average Mean Squared Error (MSE): {avg_mse:.4f}")
    print(f"Average R²: {avg_r2:.4f}")
    print("\nCoefficients:\n", results_df.to_string(index=False))

    return results_df


##  w/Regularization

In [ ]:
def lasso_pipeline(df, count_distance):
    df = process_data(df, True, True)
    X = X_subset(df, count_distance, True)
    y = df['Price']

    lasso_model = make_pipeline(Lasso(max_iter=10000, fit_intercept=True))  # Increased max_iter for convergence

    alpha_values = np.linspace(0.01, 1, 100)

    lasso_parameters = {'lasso__alpha': alpha_values}  
    lasso_reg = GridSearchCV(
        estimator=lasso_model, 
        param_grid=lasso_parameters,
        scoring=['neg_mean_squared_error', 'r2'],
        refit='neg_mean_squared_error', 
        cv=5
    )
    lasso_reg.fit(X, y)

    best_alpha = lasso_reg.best_params_['lasso__alpha']
    best_mse = -lasso_reg.best_score_ 
    best_index = lasso_reg.best_index_
    best_r2 = lasso_reg.cv_results_['mean_test_r2'][best_index]  
    best_coef = lasso_reg.best_estimator_.named_steps['lasso'].coef_

    coef_df = pd.DataFrame({"Feature": X.columns, "Coefficient": best_coef})
    coef_df["Coefficient"] = coef_df["Coefficient"].apply(lambda x: f"{x:.4f}")  # Round to 4 decimals

    print("\n===== Lasso Regression Results =====")
    print(f"Best Alpha: {best_alpha:.4f}")
    print(f"Best MSE: {best_mse:.4f}")
    print(f"Best R²: {best_r2:.4f}")
    print("\nBest Coefficients:\n", coef_df.to_string(index=False))

    return lasso_reg


In [ ]:
def ridge_pipeline(df, count_distance):
    df = process_data(df, True, True)
    X = X_subset(df, count_distance, True)
    y = df['Price']

    ridge_model = make_pipeline(Ridge(max_iter=10000, fit_intercept=True))  # Increased max_iter for convergence

    alpha_values = np.linspace(0.01, 1, 100)

    ridge_parameters = {'ridge__alpha': alpha_values}  
    ridge_reg = GridSearchCV(
        estimator=ridge_model, 
        param_grid=ridge_parameters,
        scoring=['neg_mean_squared_error', 'r2'],
        refit='neg_mean_squared_error', 
        cv=5
    )
    ridge_reg.fit(X, y)

    best_alpha = ridge_reg.best_params_['ridge__alpha']
    best_mse = -ridge_reg.best_score_ 
    best_index = ridge_reg.best_index_
    best_r2 = ridge_reg.cv_results_['mean_test_r2'][best_index]  
    best_coef = ridge_reg.best_estimator_.named_steps['ridge'].coef_

    coef_df = pd.DataFrame({"Feature": X.columns, "Coefficient": best_coef})
    coef_df["Coefficient"] = coef_df["Coefficient"].apply(lambda x: f"{x:.4f}")  # Round to 4 decimals

    print("\n===== ridge Regression Results =====")
    print(f"Best Alpha: {best_alpha:.4f}")
    print(f"Best MSE: {best_mse:.4f}")
    print(f"Best R²: {best_r2:.4f}")
    print("\nBest Coefficients:\n", coef_df.to_string(index=False))

    return ridge_reg


### Archived?

In [ ]:
# SPECIFIC N
def basic_model2_with_feature_selection(regularization_type, alpha, df, count_distance, remove_outliers, normalize_data, n_features_to_select):
    df = process_data(df, remove_outliers, normalize_data)
    y = df['Price']
    X = X_subset(df, count_distance, True)
    X = sm.add_constant(X)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    if regularization_type == "ridge": 
        pipeline = Pipeline([('scaler', StandardScaler()),('model', Ridge(alpha=alpha))])

        # Use RFE for feature selection
        rfe = RFE(estimator=Ridge(alpha=alpha), n_features_to_select=n_features_to_select)
        rfe.fit(X_train, y_train)

    if regularization_type == "lasso":
        pipeline = Pipeline([('scaler', StandardScaler()), ('model', Lasso(alpha=alpha))])

        # Use RFE for feature selection
        rfe = RFE(estimator=Lasso(alpha=alpha), n_features_to_select=n_features_to_select)
        rfe.fit(X_train, y_train)

    # Identify selected features and their ranking from most important to least important
    selected_features = X.columns[rfe.ranking_]

    # Fit pipeline only on selected features
    X_train_selected = X_train[selected_features]
    X_test_selected = X_test[selected_features]

    pipeline.fit(X_train_selected, y_train)

    # Predict and calculate R^2
    y_pred = pipeline.predict(X_test_selected)
    r2 = r2_score(y_test, y_pred)

    # Feature importance (coefficients may not be directly interpretable with Ridge)
    coefficients = pipeline.named_steps['model'].coef_
    feature_importance = dict(zip(selected_features, coefficients))

    # print(f"Most Important Features (RFE Ranking):")
    # for i, feature in enumerate(selected_features):
    #     print(f"{i+1}. {feature}")  # Print features with ranking (1 being most important)
    print(f"R-squared (R²): {r2:.4f}")
    print(feature_importance)
    return pipeline, r2, feature_importance

In [ ]:
# CHECK DIFFERENT N

def basic_model2_with_feature_selection2(regularization_type, alpha, df, count_distance, remove_outliers, normalize_data, n_features_to_select):
    df = process_data(df, remove_outliers, normalize_data)
    y = df['Price']
    X = X_subset(df, count_distance, True)
    X = sm.add_constant(X)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    if regularization_type == "ridge": 
        pipeline = Pipeline([('scaler', StandardScaler()),('model', Ridge(alpha=alpha))])
    if regularization_type == "lasso":
        pipeline = Pipeline([('scaler', StandardScaler()), ('model', Lasso(alpha=alpha))])

    # Use RFE for feature selection and evaluate for different n_features_to_select values
    best_r2 = -float('inf')
    best_n_features = None
    best_pipeline = None
    best_selected_features = None
    best_coefficients = None

    for n in range(1, X_train.shape[1] + 1):  # Try all possible values of n_features_to_select
        if regularization_type == "ridge":
            rfe = RFE(estimator=Ridge(alpha=alpha), n_features_to_select=n)
        if regularization_type == "lasso":
            rfe = RFE(estimator=Lasso(alpha=alpha), n_features_to_select=n)
        rfe.fit(X_train, y_train)
        
        selected_features = X.columns[rfe.support_]  # Use the support mask to select the features
        X_train_selected = X_train[selected_features]
        X_test_selected = X_test[selected_features]

        # Fit pipeline only on selected features
        pipeline.fit(X_train_selected, y_train)

        # Predict and calculate R²
        y_pred = pipeline.predict(X_test_selected)
        r2 = r2_score(y_test, y_pred)

        if r2 > best_r2:  # Update the best R2 and corresponding n
            best_r2 = r2
            best_n_features = n
            best_pipeline = pipeline
            best_selected_features = selected_features
            best_coefficients = pipeline.named_steps['model'].coef_

    # Rank the features based on the absolute value of their coefficients
    feature_importance = pd.DataFrame({
        'Feature': best_selected_features,
        'Coefficient': best_coefficients
    })

    feature_importance['Abs_Coefficient'] = feature_importance['Coefficient'].abs()
    feature_importance = feature_importance.sort_values(by='Abs_Coefficient', ascending=False)

    print(f"Best R-squared (R²): {best_r2:.4f} with {best_n_features} features selected")
    print("\nRanked Features (Based on coefficient):")
    for idx, row in feature_importance.iterrows():
        print(f"{row['Feature']}: {row['Coefficient']:.4f}")

    return best_pipeline, best_r2, feature_importance

In [ ]:
def basic_model2_with_feature_selection3(regularization_type, alpha_range, df, count_distance, remove_outliers, normalize_data, n_features_to_select):
    # Preprocess data
    df = process_data(df, remove_outliers, normalize_data)
    y = df['Price']
    X = X_subset(df, count_distance, True)
    X = sm.add_constant(X)

    # Regularization model selection
    if regularization_type == "ridge":
        base_model = Ridge()
    elif regularization_type == "lasso":
        base_model = Lasso()

    # Hyperparameter tuning for alpha using grid search
    param_grid = {'alpha': alpha_range}
    grid_search = GridSearchCV(estimator=base_model, param_grid=param_grid, cv=5, scoring='neg_mean_squared_error')
    grid_search.fit(X, y)

    # Best alpha and corresponding performance
    best_alpha = grid_search.best_params_['alpha']
    best_neg_mse = grid_search.best_score_
    best_mse = -best_neg_mse
    print(f"Best Alpha: {best_alpha:.4f}")
    print(f"Best MSE from GridSearchCV: {best_mse:.4f}")

    # RFE for feature selection
    rfe = RFE(estimator=base_model.set_params(alpha=best_alpha), n_features_to_select=n_features_to_select)
    rfe.fit(X, y)

    # Selected features
    selected_features = X.columns[rfe.support_]
    # print(f"Selected Features: {list(selected_features)}")

    # Cross-validation setup
    cv = KFold(n_splits=5, shuffle=True, random_state=42)

    # Cross-validated predictions
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model', base_model.set_params(alpha=best_alpha))
    ])
    y_pred_cv = cross_val_predict(pipeline, X[selected_features], y, cv=cv)

    # Metrics
    r2_cv = r2_score(y, y_pred_cv)
    mse_cv = mean_squared_error(y, y_pred_cv)

    # Permutation importance for feature ranking (cross-validated)
    pipeline.fit(X[selected_features], y)  # Fit on the entire dataset
    perm_importance = permutation_importance(
        pipeline, X[selected_features], y, scoring='neg_mean_squared_error', n_repeats=10, random_state=42
    )
    feature_importance = {feature: importance for feature, importance in zip(selected_features, perm_importance.importances_mean)}

    # Sort features by importance
    ranked_features = sorted(feature_importance.items(), key=lambda x: x[1], reverse=True)

    print(f"Cross-Validated R-squared (R²): {r2_cv:.4f}")
    print(f"Cross-Validated Mean Squared Error (MSE): {mse_cv:.4f}")
    print("Feature Importance (Ranked by MSE):")
    for feature, importance in ranked_features:
        print(f"{feature}: {importance:.4f}")

    return pipeline, r2_cv, mse_cv, ranked_features

In [ ]:
def plot_lasso_coefficients_over_alpha(df_name, alpha_range, df, count_distance, remove_outliers, normalize_data):
    # Preprocess data
    df = process_data(df, remove_outliers, normalize_data)
    
    y = df['Price']
    X = X_subset(df, count_distance, True)
    X = sm.add_constant(X)

    # Store coefficients for each alpha
    coefficients = {feature: [] for feature in X.columns}
    
    for alpha in alpha_range:
        lasso = Lasso(alpha=alpha)
        lasso.fit(X, y)
        
        # Store coefficients for each feature
        for feature, coef in zip(X.columns, lasso.coef_):
            coefficients[feature].append(coef)

    # Plot coefficients as a function of alpha
    plt.figure(figsize=(9, 6))
    for feature, coef_values in coefficients.items():
        plt.plot(alpha_range, coef_values, label=feature)

    plt.xlabel("Alpha")
    plt.ylabel("Coefficient Value")
    plt.title(f"{df_name} - Lasso Coefficients vs. Alpha")
    plt.legend(loc="best", bbox_to_anchor=(1.05, 1))
    plt.tight_layout()
    plt.savefig(f"{df_name} - Lasso Coefficients vs. Alpha.png")
    plt.show()
    plt.close